## **CAKE** (**C**onfidence in **A**ssignments via **K**-partition **E**nsembles)

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_samples, confusion_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import LabelEncoder

def sil_samples(X, labels, approximation=False, centers=None):
    """
    Compute silhouette scores for each point in the dataset,
    with approximate fast centroid-based computation option.
    """
    # Ensure arrays
    X = np.asarray(X)
    labels = np.asarray(labels)
    if X.ndim != 2:
       raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
    if labels.ndim != 1 or labels.shape[0] != X.shape[0]:
       raise ValueError("labels must be a 1D array of length n_samples.")

    unique_labels, inv = np.unique(labels, return_inverse=True)
    k = unique_labels.size
    if k < 2:
       raise ValueError("Silhouette computation requires at least 2 clusters.")

    # Exact silhouette scores
    if approximation == False:
       silhouette_scores = silhouette_samples(X, labels=labels)
       return silhouette_scores

    # Centroid-based approximate silhouette scores
    n_samples, n_features = X.shape

    if centers is None:
       centers = np.array([X[inv == i].mean(axis=0) for i in range(k)], dtype=float)
    else:
       centers = np.asarray(centers, dtype=float)
       if centers.ndim != 2 or centers.shape[1] != n_features:
          raise ValueError(f"centers must have shape (k, d) with d={n_features}.")
       if centers.shape[0] != k:
          raise ValueError(f"centers.shape[0] must equal number of clusters k={k}.")
       if not np.array_equal(unique_labels, np.arange(k)):
          raise ValueError("When passing ndarray centers, labels must be dense 0..k-1.")

    # Squared distances to all centroids
    D_sq = euclidean_distances(X, centers, squared=True)

    # a(i): distance to own centroid
    a = np.sqrt(np.maximum(D_sq[np.arange(n_samples), inv], 0.0))

    # b(i): distance to nearest other centroid
    D_sq[np.arange(n_samples), inv] = np.inf
    b = np.sqrt(np.min(D_sq, axis=1))

    # Silhouette per point
    denom = np.maximum(np.maximum(a, b), 1e-12)
    s_point = (b - a) / denom

    # Singleton clusters -> silhouette = 0
    counts = np.bincount(inv, minlength=k).astype(int)
    s_point[counts[inv] < 2] = 0.0

    silhouette_scores = np.clip(s_point, -1.0, 1.0)

    return silhouette_scores

def sil_samples_stats(X, labels_list, approximation=False, centers_list=None):
    """
    Compute and aggregate silhouette scores across multiple clustering runs,
    returning per-sample mean and standard deviation of silhouette scores.
    """
    X = np.asarray(X)
    n_samples = X.shape[0]
    n_runs = len(labels_list)
    if n_runs < 2:
       raise ValueError("Clustering Ensemble must contain at least 2 partitions.")

    if centers_list is not None and len(centers_list) != n_runs:
       raise ValueError("centers_list must match the number of labelings in labels_list")

    # Initialize matrix to hold silhouette scores
    sil_scores = np.zeros((n_runs, n_samples), dtype=float)

    for i, labels in enumerate(labels_list):
        labels_arr = np.asarray(labels)
        centers = None
        if approximation and centers_list is not None:
           centers = centers_list[i]

        # Compute silhouette scores for this run
        sil_scores[i] = sil_samples(X, labels_arr, approximation=approximation, centers=centers)

    # Compute mean and standard deviation of sample-silhouette scores over the ensemble
    mean_sil_samples = sil_scores.mean(axis=0)
    std_sil_samples = sil_scores.std(axis=0)

    return mean_sil_samples, std_sil_samples

def align_labels(target, source):
    """
    Aligns the labels in "source" to match the labels in "target"
    using the Hungarian Algorithm based on a contingency matrix.

    Parameters:
    - target: array-like of shape (n_samples,)
              The reference label vector to align to.
    - source: array-like of shape (n_samples,)
              The label vector to be permuted for alignment.

    Returns:
    - aligned: np.ndarray of shape (n_samples,)
               The source labels, remapped to best match the target labels.
    """
    target = np.asarray(target)
    source = np.asarray(source)
    unique_target = np.unique(target)
    unique_source = np.unique(source)
    if len(unique_target) != len(unique_source):
       raise ValueError(
           f"Cannot align: Target has {len(unique_target)} clusters {unique_target.tolist()}, "
           f"but source has {len(unique_source)} clusters {unique_source.tolist()}"
       )
    # Encode labels to indices based on their unique values
    le_target = LabelEncoder().fit(unique_target)
    le_source = LabelEncoder().fit(unique_source)

    target_encoded = le_target.transform(target)
    source_encoded = le_source.transform(source)

    # Confusion matrix with indices aligned to encoded labels
    c_matrix = confusion_matrix(target_encoded, source_encoded)

    # Apply the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(-c_matrix)

    # Create mapping from source labels to target labels
    mapping = {
        le_source.classes_[src_col]: le_target.classes_[tgt_row]
        for tgt_row, src_col in zip(row_ind, col_ind)
    }
    # Remap source labels using the mapping
    aligned = np.vectorize(mapping.get)(source)

    return aligned

def pairwise_stability(labels_runs):
    """
    Computes per-point clustering stability across multiple runs by aligning labels
    pairwise using the Hungarian algorithm to account for label permutations.

    Parameters:
    - labels_runs: list of array-like
        List of label arrays from multiple clustering runs. Each array must have the
        same number of samples and clusters (unique labels).

    Returns:
    - stability: np.ndarray of shape (n_samples,)
        Per-point stability score between 0 and 1, indicating the fraction of run-pairs
        where the point's label matches after optimal alignment.

    Notes:
    - Requires all runs to have the same number of clusters (unique labels).
    - Label alignment is done pairwise between runs using the Hungarian algorithm.
    - High stability (~1) indicates stable cluster assignments across runs.
    """
    labels_runs = [np.asarray(labels) for labels in labels_runs]
    n_runs, n_samples = len(labels_runs), len(labels_runs[0])
    if n_runs < 2:
       raise ValueError("pairwise_stability needs at least 2 runs.")

    # Counters: How many run-pairs agree per point
    agreement_counts = np.zeros(n_samples, dtype=int)
    total_pairs = 0

    # For every unique pair of runs
    for r1 in range(n_runs):
        labels_r1 = labels_runs[r1]
        for r2 in range(r1+1, n_runs):
            labels_r2 = labels_runs[r2]

            # Align labels_r2 to labels_r1 using Hungarian
            aligned_r2 = align_labels(labels_r1, labels_r2)
            matches = (labels_r1 == aligned_r2)

            # Add matches to total per-point agreement count
            agreement_counts += matches
            total_pairs += 1

    return agreement_counts/total_pairs

def cake(X, labels_list, method='product', approximation=False, centers_list=None, geom_norm='clip'):
    """
    Compute a confidence score per point for clustering ensembles, defined as:
    stability_i * geometric_stability_i or
    2 * stability_i * geometric_stability_i / [stability_i + geometric_stability_i]
    where stability is the pairwise label agreement across runs,
    and mean_sil_i, std_sil_i are the statistics from sil_samples_stats.

    Parameters:
    - X: array-like, shape (n_samples, n_features)
    - labels_list: list of array-like, each of shape (n_samples,)
        A list of cluster labelings for the same dataset X.
    - approximation: bool, default=False
        Whether to use the approximate silhouette computation.
    - centers_list: list of pd.Series or array-like, optional
        If approximation=True, an optional list of centroids for each clustering.
    - method: str, default='product'
        'product' or 'harmonic_mean' for the formulation of the CAKE scores.
    - geo_norm: str, default='clip'
        - 'affine' (default): maps geom_raw in [-1,1] to [0,1] via (x+1)/2 (preserves negatives).
        - 'clip': max(mean_sil - std_sil, 0) clipped at 1 (original behavior; discards negatives).

    Returns:
    - cake_scores: np.ndarray, shape (n_samples,)
        The CAKE scores for each point.
    - stability: np.ndarray, shape (n_samples,)
        The stability scores for each point.
    - geom_stability: np.ndarray, shape (n_samples,)
        The silhouette-based reliability scores for each point (mean_sil_i - std_sil_i).
    - summary: pd.DataFrame
        Summary of above metrics per point in X.
    """
    # Compute silhouette statistics
    mean_sil, std_sil = sil_samples_stats(X, labels_list,
                                          approximation=approximation,
                                          centers_list=centers_list)

    # Assignment stability across the ensemble
    stability = pairwise_stability(labels_list)

    # Silhouette-based stability across the ensemble
    geom_raw = mean_sil - std_sil
    if geom_norm == 'clip':
       geom_stability = np.clip(mean_sil - std_sil, 0.0, 1.0)
    else:
       geom_stability = np.clip((geom_raw + 1) / 2, 0, 1) # preserves information for negative scores

    # Calculate confidence scores
    if method == 'product':
       cake_scores = stability * geom_stability
    elif method == 'harmonic_mean':
       num = 2.0 * stability * geom_stability
       denom = np.maximum(stability + geom_stability, 1e-8)
       cake_scores = num / denom
    else:
       raise ValueError(
           f"Unknown method: {method}. Use 'product' or 'harmonic_mean'."
       )

    # Summary DataFrame
    summary = pd.DataFrame({
        'Mean Silhouette': mean_sil,
        'STD Silhouette': std_sil,
        'Geometric Stability': geom_stability,
        'Assignment Stability': stability,
        'CAKE': cake_scores
    })

    return cake_scores, stability, geom_stability, summary

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from collections import OrderedDict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    load_digits,
    fetch_openml,
    fetch_20newsgroups
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from tensorflow.keras.datasets import fashion_mnist
from scipy.stats import t
from sklearn.metrics import f1_score, adjusted_mutual_info_score
from scipy.stats import kendalltau, spearmanr
from sklearn.mixture import GaussianMixture
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.decomposition import PCA
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
import time, gc

def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    accuracy = cm[row_ind, col_ind].sum() / np.sum(cm)
    return accuracy

def micro_sil(X, labels, approximation=False, centers=None):
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    return float(np.mean(s))

def macro_sil(X, labels, approximation=False, centers=None):
    labels = np.asarray(labels)
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    uniq = np.unique(labels)
    cluster_means = [s[labels == lab].mean() for lab in uniq]
    return float(np.mean(cluster_means))

def consensus_medoid_majority(labels_runs):
    if not labels_runs:
        raise ValueError("labels_runs must be a non-empty list of 1D arrays.")
    labs = [np.asarray(l) for l in labels_runs]
    R = len(labs)
    n = labs[0].shape[0]
    if any(len(l) != n for l in labs):
        raise ValueError("All runs must have the same number of samples.")

    ks = {np.unique(l).size for l in labs}
    if len(ks) != 1:
        raise ValueError(f"All runs must have the same number of clusters; got {sorted(ks)}.")

    # pairwise symmetric Hamming distances after alignment
    D = np.zeros((R, R), dtype=float)
    for i in range(R):
        li = labs[i]
        for j in range(i+1, R):
            lj = labs[j]
            aligned_j = align_labels(li, lj)
            d_ij = 1.0 - np.mean(aligned_j == li)
            aligned_i = align_labels(lj, li)
            d_ji = 1.0 - np.mean(aligned_i == lj)
            D[i, j] = D[j, i] = 0.5 * (d_ij + d_ji)

    medoid = int(np.argmin(D.sum(axis=1)))
    ref = labs[medoid]
    aligned = [align_labels(ref, l) for l in labs]   # align all runs to the medoid
    A = np.vstack(aligned)                           # (R, n)

    # majority vote per point; tie -> medoid label
    z_star = np.empty(n, dtype=ref.dtype)
    for i in range(n):
        counts = Counter(A[:, i])
        top = counts.most_common(2)
        if len(top) == 1 or top[0][1] > top[1][1]:
            z_star[i] = top[0][0]
        else:
            z_star[i] = ref[i]  # tie-break to medoid label

    agree = np.mean(A == z_star, axis=0)
    return z_star, agree

## Datasets

In [3]:
def load_synthetic_blobs():
    X, y = make_blobs(
        n_samples=4000,
        centers=[[0, 0], [5, 5], [12, 0]],
        cluster_std=2,
        random_state=42
    )
    return X, y

def load_clusters_with_noise():
    X_clusters, y_clusters = make_blobs(
        n_samples=3000, centers=[[0, 0], [5, 5], [10, 0]],
        cluster_std=1.0, random_state=10
    )
    rng = np.random.default_rng(42)
    X_noise = np.random.uniform(low=-10, high=15, size=(1500, 2))
    y_noise = [-1] * 1500  # Label -1 for noise

    X = np.vstack([X_clusters, X_noise])
    y = np.concatenate([y_clusters, y_noise])
    return X, y

def load_square_ring():
    centers = [[-3, -3], [-3, 3], [3, -3], [3, 3]]
    stds = [1.9] * 4
    X_list, y_list = [], []

    for i, c in enumerate(centers):
        X_i, _ = make_blobs(n_samples=750, centers=[c], cluster_std=stds[i], random_state=20+i)
        X_list.append(X_i)
        y_list.extend([i]*750)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_synthetic_overlap():
    from sklearn.datasets import make_blobs
    import numpy as np

    centers = [[0, 0], [2, 0], [8, 8]]
    stds = [2, 2.3, 1.5]

    X_list, y_list = [], []
    for i, (c, s) in enumerate(zip(centers, stds)):
        X_i, _ = make_blobs(n_samples=1000, centers=[c], cluster_std=s, random_state=42+i)
        X_list.append(X_i)
        y_list.extend([i] * 1000)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_high_density_contrast():
    centers = [[0, 0], [8, 0], [4, 8]]
    stds = [0.2, 3.0, 1.0]
    sizes = [1000, 2000, 1000]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=100+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_overlap_low_density():
    centers = [[0, 0], [4, 0], [8, 0]]
    stds = [0.4, 2.5, 0.4]
    sizes = [1000, 2000, 1000]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=200+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

def load_imbalanced_blobs():
    centers = [[0, 0], [5, 5], [10, 0]]
    stds = [0.3, 1.5, 2.5]
    sizes = [500, 1000, 2500]

    X_list, y_list = [], []
    for i, (c, s, n) in enumerate(zip(centers, stds, sizes)):
        X_i, _ = make_blobs(n_samples=n, centers=[c], cluster_std=s, random_state=400+i)
        X_list.append(X_i)
        y_list.extend([i] * n)

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y

loaders_synth = {
    'S1':    load_synthetic_blobs,
    'S2':    load_synthetic_overlap,
    'S3':    load_clusters_with_noise,
    'S4':    load_square_ring,
    'S5':    load_high_density_contrast,
    'S6':    load_overlap_low_density,
    'S7':    load_imbalanced_blobs
}

In [4]:
# 20 News Groups
def load_20ng_bert(n_components=100):
    all_categories = [
            'alt.atheism',
            'comp.graphics',
            'comp.os.ms-windows.misc',
            'comp.sys.ibm.pc.hardware',
            'comp.sys.mac.hardware',
            'comp.windows.x',
            'misc.forsale',
            'rec.autos',
            'rec.motorcycles',
            'rec.sport.baseball',
            'rec.sport.hockey',
            'sci.crypt',
            'sci.electronics',
            'sci.med',
            'sci.space',
            'soc.religion.christian',
            'talk.politics.guns',
            'talk.politics.mideast',
            'talk.politics.misc',
            'talk.religion.misc']
    subset = fetch_20newsgroups(subset='all', categories=all_categories, remove=('headers', 'footers', 'quotes'))
    texts = subset.data
    labels = subset.target

    model = SentenceTransformer('all-MiniLM-L6-v2')
    X = model.encode(texts, show_progress_bar=True)

    X_scaled = StandardScaler().fit_transform(X)
    X_pca = PCA(n_components=n_components, random_state=42).fit_transform(X_scaled)

    return X_pca, labels

# Pendigits
def load_pendigits():
    data = fetch_openml('pendigits', version=1, as_frame=False)
    X = data.data
    y = LabelEncoder().fit_transform(data.target)
    return X, y

# Letter
def load_letter():
    data = fetch_openml('letter', version=1, as_frame=False)
    X = data.data
    y = LabelEncoder().fit_transform(data.target)
    X = StandardScaler().fit_transform(X)
    return X, y

# Fashion Mnist
def load_fashion_mnist_scaled():
    (X_train, y_train), _ = fashion_mnist.load_data()
    X = X_train.reshape((X_train.shape[0], -1)).astype(np.float32)
    return X, y_train

# Satimage
def load_satimage():
    X, y = fetch_openml('satimage', version=1, return_X_y=True, as_frame=False)
    y = LabelEncoder().fit_transform(y)
    pca = PCA(n_components=30, random_state=42)
    X_pca = pca.fit_transform(X)
    return X_pca, y

# Breast Cancer
def load_breast_cancer_processed():
    X, y = load_breast_cancer(return_X_y=True)
    X = StandardScaler().fit_transform(X)
    X_pca = PCA(n_components=10, random_state=42).fit_transform(X)
    return X_pca, y

loaders_real = {
    'iris':           lambda: load_iris(return_X_y=True),
    'breast_cancer':  load_breast_cancer_processed,
    'digits':         lambda: load_digits(return_X_y=True),
    '20newsgroups':   load_20ng_bert,
    'fashionmnist':   load_fashion_mnist_scaled,
    'pendigits':      load_pendigits,
    'letter':         load_letter,
    'satimage':       load_satimage
}

## Comparison with FCM, LOF, GMM

In [5]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.impute import SimpleImputer

def infer_k_from_y(y, ignore_noise_label=True):
    y = np.asarray(y)
    if ignore_noise_label:
        uniq = np.unique(y[y != -1])
    else:
        uniq = np.unique(y)
    return len(uniq)

def align_pred_to_truth(y_true, y_pred):
    """
    Hungarian-align predicted cluster labels to ground-truth labels.
    Works even when the number of predicted clusters and true classes differ.
    Returns aligned predictions and binary correctness labels.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    le_true = LabelEncoder().fit(y_true)
    le_pred = LabelEncoder().fit(y_pred)

    yt = le_true.transform(y_true)
    yp = le_pred.transform(y_pred)

    n_true = len(le_true.classes_)
    n_pred = len(le_pred.classes_)

    cm = np.zeros((n_true, n_pred), dtype=int)
    np.add.at(cm, (yt, yp), 1)

    row_ind, col_ind = linear_sum_assignment(-cm)

    mapping = {
        le_pred.classes_[c]: le_true.classes_[r]
        for r, c in zip(row_ind, col_ind)
    }

    aligned = np.array([mapping.get(lbl, None) for lbl in y_pred], dtype=object)
    correct = np.array([int(a == t) for a, t in zip(aligned, y_true)], dtype=int)

    return aligned, correct, mapping, cm

def safe_auroc(y_true_binary, scores):
    y_true_binary = np.asarray(y_true_binary)
    if len(np.unique(y_true_binary)) < 2:
        return np.nan
    return roc_auc_score(y_true_binary, scores)

def entropy_confidence(P, eps=1e-12):
    """
    P: (n_samples, k) row-stochastic memberships/posteriors
    returns: 1 - normalized entropy in [0,1]
    """
    P = np.asarray(P, dtype=float)
    P = np.clip(P, eps, 1.0)
    P = P / P.sum(axis=1, keepdims=True)
    k = P.shape[1]
    H = -np.sum(P * np.log(P), axis=1)
    return 1.0 - (H / np.log(k))

def margin_confidence(P):
    """
    Top-1 minus top-2 confidence margin.
    """
    P = np.asarray(P, dtype=float)
    P_sorted = np.sort(P, axis=1)
    return P_sorted[:, -1] - P_sorted[:, -2]


def fit_fcm(X, k, m=2.0, max_iter=200, tol=1e-5, random_state=42):
    """
    Basic Fuzzy C-Means.
    Returns centers, membership matrix U (n x k), hard labels.
    """
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    rng = np.random.default_rng(random_state)

    U = rng.random((n, k))
    U = U / U.sum(axis=1, keepdims=True)

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + 1e-12)

        D = euclidean_distances(X, centers)
        D = np.maximum(D, 1e-12)

        zero_mask = D <= 1e-10
        U_new = np.zeros_like(U)

        rows_with_zero = zero_mask.any(axis=1)
        if np.any(rows_with_zero):
            U_new[rows_with_zero] = (
                zero_mask[rows_with_zero] /
                zero_mask[rows_with_zero].sum(axis=1, keepdims=True)
            )

        rows_no_zero = ~rows_with_zero
        if np.any(rows_no_zero):
            power = 2.0 / (m - 1.0)
            ratio = (D[rows_no_zero, :, None] / D[rows_no_zero, None, :]) ** power
            U_new[rows_no_zero] = 1.0 / ratio.sum(axis=2)

        U = U_new

        if np.max(np.abs(U - U_old)) < tol:
            break

    labels = np.argmax(U, axis=1)
    return centers, U, labels


def build_kmeans_ensemble(X, k, R=20, random_state=42):
    """
    Builds the default CAKE ensemble:
    random-seed KMeans runs with n_init=1 and random init.
    Returns labels_list and centers_list.
    """
    labels_list = []
    centers_list = []

    for r in range(R):
        km = KMeans(
            n_clusters=k,
            init='random',
            n_init=1,
            random_state=random_state + r
        )
        labels = km.fit_predict(X)
        labels_list.append(labels)
        centers_list.append(km.cluster_centers_)

    return labels_list, centers_list

def consensus_correctness_target(X, y, R=20, random_state=42):
    """
    Main target:
    correctness of the KMeans-ensemble consensus partition z*.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    k = infer_k_from_y(y, ignore_noise_label=True)

    labels_list, centers_list = build_kmeans_ensemble(
        X, k=k, R=R, random_state=random_state
    )

    z_star, agree = consensus_medoid_majority(labels_list)
    aligned_consensus, correct, mapping, cm = align_pred_to_truth(y, z_star)

    return {
        "target_name": "Consensus correctness",
        "k": k,
        "labels_list": labels_list,
        "centers_list": centers_list,
        "z_star": z_star,
        "consensus_agreement": agree,
        "aligned_consensus": aligned_consensus,
        "correct": correct,
        "mapping": mapping,
        "confusion": cm
    }

def kmeans_cake_hm_from_precomputed(X, labels_list, centers_list, approximation=True):
    cake_hm, _, _, _ = cake(
        X,
        labels_list,
        method='harmonic_mean',
        approximation=approximation,
        centers_list=centers_list,
        geom_norm='clip'
    )
    return cake_hm


def _cluster_local_lof_confidence_single_run(
    X,
    labels,
    lof_neighbors=20,
    neutral_confidence=0.5
):
    """
    Compute cluster-local LOF confidence for one clustering run.

    For each point:
      - consider only the points in its assigned cluster
      - compute LOF inside that cluster
      - transform LOF to an inlier-confidence in (0, 1]

    If a cluster is too small for LOF, assign a neutral confidence.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    n = len(labels)

    conf = np.full(n, neutral_confidence, dtype=float)

    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        cluster_size = len(idx)

        # Need at least 3 points to make any neighborhood idea meaningful
        if cluster_size < 3:
            conf[idx] = neutral_confidence
            continue

        # LOF requires n_neighbors < cluster_size
        n_neighbors_local = min(max(2, lof_neighbors), cluster_size - 1)

        # If still invalid-> neutral
        if n_neighbors_local < 2:
            conf[idx] = neutral_confidence
            continue

        Xc = X[idx]

        lof = LocalOutlierFactor(n_neighbors=n_neighbors_local, novelty=False)
        lof.fit_predict(Xc)

        # sklearn gives negative_outlier_factor_:
        # around -1 for inliers, more negative for outliers
        nof = lof.negative_outlier_factor_
        lof_raw = -nof  # ~1 for inliers, >1 for more outlier-like points

        # Convert to bounded inlier-confidence in (0,1]
        # inlier-ish: lof_raw ~ 1 => conf ~ 1
        # outlier-ish: lof_raw > 1 => conf decreases
        conf_local = 1.0 / np.maximum(1.0, lof_raw)

        conf[idx] = conf_local

    return conf

def ensemble_cluster_local_lof_confidence(
    X,
    labels_list,
    lof_neighbors=20,
    neutral_confidence=0.5
):
    """
    Average cluster-local LOF confidence across clustering runs.
    """
    X = np.asarray(X, dtype=float)
    labels_list = [np.asarray(lbl) for lbl in labels_list]

    per_run_scores = []
    for labels in labels_list:
        s = _cluster_local_lof_confidence_single_run(
            X,
            labels,
            lof_neighbors=lof_neighbors,
            neutral_confidence=neutral_confidence
        )
        per_run_scores.append(s)

    return np.mean(np.vstack(per_run_scores), axis=0)


def build_main_score_bank(X, y, labels_list, centers_list, random_state=42, lof_neighbors=20):
    """
    Main comparison:
      - CAKE(HM) from the KMeans ensemble
      - FCM max-membership / entropy
      - global LOF inlier-confidence
      - ensemble cluster-local LOF confidence
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    k = infer_k_from_y(y, ignore_noise_label=True)

    scores = {}

    # CAKE(HM) from the same KMeans ensemble used to define the consensus target
    #scores["CAKE(HM)"] = kmeans_cake_hm_from_precomputed(
    #    X, labels_list=labels_list, centers_list=centers_list, approximation=True
    #)

    # FCM native soft-confidence scores
    _, U_fcm, _ = fit_fcm(X, k=k, m=2.0, max_iter=200, tol=1e-5, random_state=random_state)
    scores["FCM max-membership"] = np.max(U_fcm, axis=1)
    scores["FCM entropy-confidence"] = entropy_confidence(U_fcm)

    # Global LOF density-based inlier confidence
    n_neighbors = min(max(5, lof_neighbors), len(X) - 1)
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=False)
    lof.fit_predict(X)
    scores["LOF inlier-confidence"] = lof.negative_outlier_factor_

    # ensemble cluster-local LOF confidence
    scores["Ensemble cluster-local LOF"] = ensemble_cluster_local_lof_confidence(
        X,
        labels_list=labels_list,
        lof_neighbors=lof_neighbors,
        neutral_confidence=0.5
    )

    return scores


#  GMM experiment
def gmm_reference_correctness(X, y, random_state=42):
    """
    Appendix target:
    c_i = 1 if GMM assignment is correct after Hungarian alignment, else 0
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    k = infer_k_from_y(y, ignore_noise_label=True)

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        n_init=3,
        reg_covar=1e-6,
        random_state=random_state
    )
    pred = gmm.fit_predict(X)

    aligned_pred, correct, mapping, cm = align_pred_to_truth(y, pred)
    return {
        "ref_model": "GMM",
        "k": k,
        "pred": pred,
        "aligned_pred": aligned_pred,
        "correct": correct,
        "mapping": mapping,
        "confusion": cm,
        "gmm_model": gmm
    }

def gmm_cake_hm_score(X, k, R=20, random_state=42, approximation=True):
    labels_list = []
    centers_list = []

    for r in range(R):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            n_init=1,
            reg_covar=1e-6,
            random_state=random_state + r
        )
        gmm.fit(X)
        labels = gmm.predict(X)
        labels_list.append(labels)
        centers_list.append(gmm.means_)

    cake_hm, _, _, _ = cake(
        X,
        labels_list,
        method='harmonic_mean',
        approximation=approximation,
        centers_list=centers_list if approximation else None,
        geom_norm='clip'
    )
    return cake_hm

def build_gmm_appendix_score_bank(X, y, R=20, random_state=42):
    """
    Appendix comparison bank for stronger GMM comparison:
      - GMM pmax / margin / entropy
      - CAKE(HM)-GMM
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    k = infer_k_from_y(y, ignore_noise_label=True)

    scores = {}

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        n_init=3,
        reg_covar=1e-6,
        random_state=random_state
    )
    gmm.fit(X)
    P_gmm = gmm.predict_proba(X)

    scores["GMM $p_{max}$"] = np.max(P_gmm, axis=1)
    scores["GMM margin"] = margin_confidence(P_gmm)
    scores["GMM entropy-confidence"] = entropy_confidence(P_gmm)
    scores["CAKE(HM)-GMM"] = gmm_cake_hm_score(
        X, k=k, R=R, random_state=random_state, approximation=True
    )

    return scores


# Evaluation wrappers

def evaluate_scores_against_target(scores_dict, correct_binary):
    rows = []
    for method, s in scores_dict.items():
        ap = average_precision_score(correct_binary, s)
        auc = safe_auroc(correct_binary, s)
        rows.append({
            "Method": method,
            "AUPRC": ap,
            "AUROC": auc
        })
    return pd.DataFrame(rows)

def format_metric_table(results_long, metric="AUPRC", method_order=None):
    tab = results_long.pivot(index="Method", columns="Dataset", values=metric)
    if method_order is not None:
        tab = tab.reindex(method_order)

    rank_cols = tab.rank(axis=0, ascending=False, method="average")
    tab["AvgRank"] = rank_cols.mean(axis=1)

    return tab.sort_values("AvgRank").round(3)

In [6]:
main_datasets = OrderedDict({
    "S1": loaders_synth["S1"],
    "S2": loaders_synth["S2"],
    "S3": loaders_synth["S3"],
    "S4": loaders_synth["S4"],
    "S5": loaders_synth["S5"],
    "S6": loaders_synth["S6"],
    "S7": loaders_synth["S7"],
    "IR": loaders_real["iris"],
    "BC": loaders_real["breast_cancer"],
    "DG": loaders_real["digits"],
    "PD": loaders_real["pendigits"],
    "LT": loaders_real["letter"],
    "SA": loaders_real["satimage"],
    "FM": loaders_real["fashionmnist"],
    "NG": loaders_real["20newsgroups"],
})

gmm_datasets = OrderedDict({
    "IR": loaders_real["iris"],
    "BC": loaders_real["breast_cancer"],
    "DG": loaders_real["digits"],
    "PD": loaders_real["pendigits"],
    "LT": loaders_real["letter"],
    "SA": loaders_real["satimage"],
    "FM": loaders_real["fashionmnist"],
    "NG": loaders_real["20newsgroups"],
})

R_ENSEMBLE = 20
SEED = 42

main_order = [
    #"CAKE(HM)",
    "FCM max-membership",
    "FCM entropy-confidence",
    "LOF inlier-confidence",
    "Ensemble cluster-local LOF",
]

gmm_order = [
    "GMM $p_{max}$",
    "GMM margin",
    "GMM entropy-confidence",
    "CAKE(HM)-GMM",
]

In [7]:
main_results = []

for ds_name, loader in main_datasets.items():
    print(f"Running main consensus-correctness experiment on {ds_name} ...")
    X, y = loader()
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    if np.isnan(X).any():
        X = SimpleImputer(strategy="mean").fit_transform(X)

    # Build consensus-correctness target from the same KMeans ensemble used by CAKE
    target = consensus_correctness_target(
        X, y,
        R=R_ENSEMBLE,
        random_state=SEED
    )
    correct = target["correct"]

    scores = build_main_score_bank(
        X, y,
        labels_list=target["labels_list"],
        centers_list=target["centers_list"],
        random_state=SEED,
        lof_neighbors=20
    )

    # Evaluate
    df_eval = evaluate_scores_against_target(scores, correct)
    df_eval["Dataset"] = ds_name
    df_eval["PositiveRate"] = correct.mean()
    main_results.append(df_eval)

main_results = pd.concat(main_results, ignore_index=True)
#display(main_results)

Running main consensus-correctness experiment on S1 ...
Running main consensus-correctness experiment on S2 ...
Running main consensus-correctness experiment on S3 ...
Running main consensus-correctness experiment on S4 ...
Running main consensus-correctness experiment on S5 ...
Running main consensus-correctness experiment on S6 ...
Running main consensus-correctness experiment on S7 ...
Running main consensus-correctness experiment on IR ...
Running main consensus-correctness experiment on BC ...
Running main consensus-correctness experiment on DG ...
Running main consensus-correctness experiment on PD ...
Running main consensus-correctness experiment on LT ...


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the numb

Running main consensus-correctness experiment on SA ...
Running main consensus-correctness experiment on FM ...
Running main consensus-correctness experiment on NG ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/589 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the numb

In [8]:
main_auprc_table = format_metric_table(
    main_results,
    metric="AUPRC",
    method_order=main_order
)

main_auroc_table = format_metric_table(
    main_results,
    metric="AUROC",
    method_order=main_order
)

print("\nAUPRC on consensus correctness:")
display(main_auprc_table)

print("\nAUROC on consensus correctness:")
display(main_auroc_table)


AUPRC on consensus correctness:


Dataset,BC,DG,FM,IR,LT,NG,PD,S1,S2,S3,S4,S5,S6,S7,SA,AvgRank
Method,,,,,,,,,,,,,,,,
FCM max-membership,0.985,0.686,0.578,0.993,0.227,0.489,0.939,0.995,0.878,0.964,0.970,0.982,0.931,0.970,0.898,1.900
FCM entropy-confidence,0.985,0.698,0.575,0.992,0.229,0.489,0.930,0.994,0.869,0.964,0.969,0.982,0.929,0.970,0.896,2.167
Ensemble cluster-local LOF,0.960,0.789,0.613,0.902,0.333,0.598,0.822,0.970,0.774,0.717,0.910,0.943,0.782,0.944,0.713,2.600
LOF inlier-confidence,0.948,0.775,0.604,0.890,0.301,0.598,0.783,0.966,0.769,0.678,0.891,0.954,0.742,0.941,0.707,3.333



AUROC on consensus correctness:


Dataset,BC,DG,FM,IR,LT,NG,PD,S1,S2,S3,S4,S5,S6,S7,SA,AvgRank
Method,,,,,,,,,,,,,,,,
FCM max-membership,0.889,0.432,0.498,0.940,0.381,0.459,0.823,0.911,0.682,0.943,0.826,0.855,0.832,0.767,0.795,1.900
FCM entropy-confidence,0.889,0.431,0.504,0.926,0.385,0.456,0.807,0.895,0.655,0.944,0.819,0.857,0.827,0.768,0.788,2.167
Ensemble cluster-local LOF,0.704,0.634,0.587,0.559,0.546,0.608,0.598,0.619,0.459,0.586,0.598,0.631,0.643,0.630,0.561,2.600
LOF inlier-confidence,0.639,0.599,0.583,0.500,0.514,0.589,0.537,0.541,0.442,0.549,0.515,0.682,0.617,0.635,0.555,3.333


**CAKE scores values were computed a previous experiment**

In [9]:
gmm_results = []

for ds_name, loader in gmm_datasets.items():
    print(f"Running GMM experiment on {ds_name} ...")
    X, y = loader()
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    if np.isnan(X).any():
        X = SimpleImputer(strategy="mean").fit_transform(X)

    # Reference correctness target from GMM
    ref = gmm_reference_correctness(X, y, random_state=SEED)
    correct = ref["correct"]

    # Scores
    scores = build_gmm_appendix_score_bank(
        X, y,
        R=R_ENSEMBLE,
        random_state=SEED
    )

    # Evaluate
    df_eval = evaluate_scores_against_target(scores, correct)
    df_eval["Dataset"] = ds_name
    df_eval["PositiveRate"] = correct.mean()
    gmm_results.append(df_eval)

gmm_results = pd.concat(gmm_results, ignore_index=True)
#display(gmm_results)

Running GMM experiment on IR ...
Running GMM experiment on BC ...
Running GMM experiment on DG ...
Running GMM experiment on PD ...
Running GMM experiment on LT ...
Running GMM experiment on SA ...
Running GMM experiment on FM ...
Running GMM experiment on NG ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/589 [00:00<?, ?it/s]

In [10]:
gmm_auprc = format_metric_table(
    gmm_results,
    metric="AUPRC",
    method_order=gmm_order
)

gmm_auroc = format_metric_table(
    gmm_results,
    metric="AUROC",
    method_order=gmm_order
)

print("AUPRC:")
display(gmm_auprc)

print("AUROC:")
display(gmm_auroc)

AUPRC:


Dataset,BC,DG,FM,IR,LT,NG,PD,SA,AvgRank
Method,,,,,,,,,
CAKE(HM)-GMM,0.947,0.967,0.684,0.996,0.444,0.639,0.924,0.806,1.750
GMM entropy-confidence,0.758,0.841,0.413,0.997,0.544,0.610,0.679,0.711,2.625
GMM $p_{max}$,0.756,0.847,0.418,0.997,0.518,0.628,0.644,0.695,2.750
GMM margin,0.756,0.848,0.418,0.997,0.516,0.638,0.638,0.692,2.875


AUROC:


Dataset,BC,DG,FM,IR,LT,NG,PD,SA,AvgRank
Method,,,,,,,,,
CAKE(HM)-GMM,0.915,0.853,0.715,0.919,0.609,0.687,0.858,0.747,1.750
GMM $p_{max}$,0.641,0.533,0.500,0.927,0.686,0.659,0.537,0.692,2.625
GMM margin,0.641,0.537,0.500,0.927,0.686,0.664,0.535,0.690,2.750
GMM entropy-confidence,0.641,0.511,0.487,0.927,0.689,0.645,0.559,0.690,2.875
